# EDA

In [1]:
import pandas as pd

df= pd.read_csv("../../data/raw/jobs.csv")
print(df.head())

   index          Job Title               Salary Estimate  \
0      0  Sr Data Scientist  $137K-$171K (Glassdoor est.)   
1      1     Data Scientist  $137K-$171K (Glassdoor est.)   
2      2     Data Scientist  $137K-$171K (Glassdoor est.)   
3      3     Data Scientist  $137K-$171K (Glassdoor est.)   
4      4     Data Scientist  $137K-$171K (Glassdoor est.)   

                                     Job Description  Rating  \
0  Description\r\n\r\nThe Senior Data Scientist i...     3.1   
1  Secure our Nation, Ignite your Future\r\n\r\nJ...     4.2   
2  Overview\r\n\r\n\r\nAnalysis Group is one of t...     3.8   
3  JOB DESCRIPTION:\r\n\r\nDo you have a passion ...     3.5   
4  Data Scientist\r\nAffinity Solutions / Marketi...     2.9   

                Company Name       Location            Headquarters  \
0         Healthfirst\r\n3.1   New York, NY            New York, NY   
1             ManTech\r\n4.2  Chantilly, VA             Herndon, VA   
2      Analysis Group\r\n3.8     Bo

### Health check 

In [2]:
print("SHAPE:", df.shape)
print("\nCOLUMNS:", df.columns.tolist())
print("\nDTYPES:\n", df.dtypes)
print("\nSTATS:\n", df.describe(include="all"))

SHAPE: (672, 15)

COLUMNS: ['index', 'Job Title', 'Salary Estimate', 'Job Description', 'Rating', 'Company Name', 'Location', 'Headquarters', 'Size', 'Founded', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors']

DTYPES:
 index                  int64
Job Title                str
Salary Estimate          str
Job Description          str
Rating               float64
Company Name             str
Location                 str
Headquarters             str
Size                     str
Founded                int64
Type of ownership        str
Industry                 str
Sector                   str
Revenue                  str
Competitors              str
dtype: object

STATS:
              index       Job Title              Salary Estimate  \
count   672.000000             672                          672   
unique         NaN             172                           30   
top            NaN  Data Scientist  $75K-$131K (Glassdoor est.)   
freq           NaN             337

### check for nulls and duplicated values

In [3]:
print("\nNULLS:\n", df.isnull().sum())
print("\nDUPLICATES:", df.duplicated().sum())


NULLS:
 index                0
Job Title            0
Salary Estimate      0
Job Description      0
Rating               0
Company Name         0
Location             0
Headquarters         0
Size                 0
Founded              0
Type of ownership    0
Industry             0
Sector               0
Revenue              0
Competitors          0
dtype: int64

DUPLICATES: 0


### check logic

In [4]:
def check_logic(df: pd.DataFrame):
    print("--- Logical Consistency Check ---")
    # Check for the common Glassdoor '-1' placeholder
    minus_ones = (df == -1).sum().sum() + (df == "-1").sum().sum()
    print(f"Total '-1' placeholders found: {minus_ones}")
    
    # Check for impossible ratings
    out_of_bounds_rating = df[(df['Rating'] < 0) | (df['Rating'] > 5)].shape[0]
    print(f"Ratings outside 0-5 range: {out_of_bounds_rating}")
    
    # Check for unrealistic years
    current_year = 2026
    future_founded = df[df['Founded'] > current_year].shape[0]
    print(f"Companies founded in the future: {future_founded}")
check_logic(df)

--- Logical Consistency Check ---
Total '-1' placeholders found: 923
Ratings outside 0-5 range: 50
Companies founded in the future: 0


In [5]:
# Convert everything to string for a unified check (handles both -1 and "-1")
placeholder_counts = (df.astype(str) == '-1').sum()

# Filter to show only columns that actually have placeholders
report = placeholder_counts[placeholder_counts > 0].sort_values(ascending=False)

print("--- Columns containing '-1' placeholders ---")
if not report.empty:
    for col, count in report.items():
        percentage = (count / len(df)) * 100
        print(f"{col:20} | Found: {count:4} | ({percentage:.1f}%)")
else:
    print("No '-1' placeholders found in any column!")

--- Columns containing '-1' placeholders ---
Competitors          | Found:  501 | (74.6%)
Founded              | Found:  118 | (17.6%)
Industry             | Found:   71 | (10.6%)
Sector               | Found:   71 | (10.6%)
Headquarters         | Found:   31 | (4.6%)
Size                 | Found:   27 | (4.0%)
Type of ownership    | Found:   27 | (4.0%)
Revenue              | Found:   27 | (4.0%)


In [6]:
# Count how many rows have -1 in the 'Founded' column
founded_minus_one_count = (df['Founded'] == -1).sum()
print(f"Number of companies with missing 'Founded' year (-1): {founded_minus_one_count}")

Number of companies with missing 'Founded' year (-1): 118


### Analyse columns values : 

In [7]:
df['Job Title'].unique()

<StringArray>
[                                                 'Sr Data Scientist',
                                                     'Data Scientist',
                           'Data Scientist / Machine Learning Expert',
                                   'Staff Data Scientist - Analytics',
                          'Data Scientist - Statistics, Early Career',
                                                       'Data Modeler',
                                         'Experienced Data Scientist',
                                          'Data Scientist - Contract',
                                                    'Data Analyst II',
                                              'Medical Lab Scientist',
 ...
 'Information Systems Engineering Specialist (Engineering Scientist)',
                 'Scientist/Research Associate-Metabolic Engineering',
            'Vice President, Biometrics and Clinical Data Management',
   'Enterprise Data Analyst (Enterprise Portfolio Manageme

In [8]:
df['Rating'].unique()

array([ 3.1,  4.2,  3.8,  3.5,  2.9,  3.9,  4.4,  3.6,  4.5,  4.7,  3.7,
        3.4,  4.1,  3.2,  4.3,  2.8,  5. ,  4.8,  3.3,  2.7,  2.2,  2.6,
        4. ,  2.5,  4.9,  2.4, -1. ,  2.3,  4.6,  3. ,  2.1,  2. ])

In [9]:
df['Type of ownership'].unique()

<StringArray>
[        'Nonprofit Organization',               'Company - Public',
        'Private Practice / Firm',              'Company - Private',
                     'Government', 'Subsidiary or Business Segment',
             'Other Organization',                             '-1',
                        'Unknown',                       'Hospital',
                  'Self-employed',           'College / University',
                       'Contract']
Length: 13, dtype: str

In [10]:

df['Industry'].unique()

<StringArray>
[                      'Insurance Carriers',
                   'Research & Development',
                               'Consulting',
    'Electrical & Electronic Manufacturing',
                  'Advertising & Marketing',
             'Computer Hardware & Software',
                'Biotech & Pharmaceuticals',
 'Consumer Electronics & Appliances Stores',
  'Enterprise Software & Network Solutions',
                              'IT Services',
                                   'Energy',
                   'Chemical Manufacturing',
                         'Federal Agencies',
                                 'Internet',
         'Health Care Services & Hospitals',
    'Investment Banking & Asset Management',
                      'Aerospace & Defense',
                                'Utilities',
                                       '-1',
                'Express Delivery Services',
                   'Staffing & Outsourcing',
          'Insurance Agencies & Brokerage

In [11]:

df['Sector'].unique()

<StringArray>
[                         'Insurance',                  'Business Services',
                      'Manufacturing',             'Information Technology',
          'Biotech & Pharmaceuticals',                             'Retail',
       'Oil, Gas, Energy & Utilities',                         'Government',
                        'Health Care',                            'Finance',
                'Aerospace & Defense',                                 '-1',
         'Transportation & Logistics',                              'Media',
                 'Telecommunications',                        'Real Estate',
                   'Travel & Tourism',             'Agriculture & Forestry',
                          'Education',                 'Accounting & Legal',
                         'Non-Profit', 'Construction, Repair & Maintenance',
                  'Consumer Services']
Length: 23, dtype: str

## Proccessing -1 values 

In [12]:
import pandas as pd
import numpy as np


def replace_negative_ones(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replaces -1 sentinel values with either:
      - np.nan  → for numeric columns and meaningful categoricals (to allow imputation later)
      - "Unknown" → for categorical columns where "Unknown" is a valid, interpretable label
      - 0       → for binary/derived columns where absence = 0

    Parameters
    ----------
    df : pd.DataFrame
        Raw jobs dataframe loaded from the CSV.

    Returns
    -------
    pd.DataFrame
        Cleaned copy of the dataframe with -1 values replaced appropriately.
    """
    df = df.copy()

    # ------------------------------------------------------------------
    # 1. NUMERIC COLUMNS → replace -1 with NaN
    #    Rationale: -1 is not a valid numeric value for rating or year.
    #    NaN allows proper median/mean imputation downstream.
    # ------------------------------------------------------------------
    numeric_minus_one = ["Rating", "Founded"]
    for col in numeric_minus_one:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].replace(-1, np.nan)

    # ------------------------------------------------------------------
    # 2. ORDINAL / STRUCTURED CATEGORICAL COLUMNS → replace -1 with "Unknown"
    #    Rationale: these columns have a natural order or meaningful set of
    #    categories. "Unknown" preserves the information that data was missing
    #    without distorting ordinal encoding or groupby aggregations.
    # ------------------------------------------------------------------
    categorical_unknown = [
        "Headquarters",       # 31 missing — geographic info, "Unknown" is interpretable
        "Size",               # 27 missing + 17 already "Unknown" — ordinal, merge buckets
        "Type of ownership",  # 27 missing — nominal, "Unknown" is a valid category
        "Industry",           # 71 missing — nominal, large cardinality
        "Sector",             # 71 missing — nominal, broader than industry
        "Revenue",            # 27 missing (on top of "Unknown / Non-Applicable") — ordinal
    ]
    for col in categorical_unknown:
        df[col] = df[col].astype(str).replace("-1", "Unknown")
        # Also unify existing "Unknown / Non-Applicable" in Revenue for consistency
        if col == "Revenue":
            df[col] = df[col].replace("Unknown / Non-Applicable", "Unknown")

    # ------------------------------------------------------------------
    # 3. HIGH-MISSINGNESS COLUMN → binarize instead of replacing
    #    Rationale: Competitors is -1 in ~75% of rows. There is little
    #    signal in the raw text; a binary flag captures the only useful info.
    # ------------------------------------------------------------------
    df["has_competitors"] = (df["Competitors"] != "-1").astype(int)
    df = df.drop(columns=["Competitors"])

    return df


# ------------------------------------------------------------------
# Bonus: helper to summarize remaining missing values after cleaning
# ------------------------------------------------------------------
def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    """Returns a summary DataFrame of missing values per column."""
    total = len(df)
    report = pd.DataFrame({
        "missing_count": df.isin([-1, "-1"]).sum() + df.isna().sum(),
        "missing_pct": ((df.isin([-1, "-1"]).sum() + df.isna().sum()) / total * 100).round(2),
        "dtype": df.dtypes,
    })
    return report[report["missing_count"] > 0].sort_values("missing_pct", ascending=False)


# ------------------------------------------------------------------
# Example usage
# ------------------------------------------------------------------
if __name__ == "__main__":
    df_raw = pd.read_csv("../../data/raw/jobs.csv")

    print("=== BEFORE CLEANING ===")
    print(f"Shape: {df_raw.shape}")

    df_clean = replace_negative_ones(df_raw)

    print("\n=== AFTER CLEANING ===")
    print(f"Shape: {df_clean.shape}")
    print("\nRemaining missing values:")
    report = missing_value_report(df_clean)
    print(report if not report.empty else "No -1 sentinel values remain.")

    print("\nSample cleaned rows:")
    print(df_clean[["Rating", "Founded", "Size", "Revenue", "Type of ownership",
                     "Industry", "Sector", "Headquarters", "has_competitors"]].head(10))

=== BEFORE CLEANING ===
Shape: (672, 15)

=== AFTER CLEANING ===
Shape: (672, 15)

Remaining missing values:
         missing_count  missing_pct    dtype
Founded            118        17.56  float64
Rating              50         7.44  float64

Sample cleaned rows:
   Rating  Founded                     Size                     Revenue  \
0     3.1   1993.0   1001 to 5000 employees                     Unknown   
1     4.2   1968.0  5001 to 10000 employees      $1 to $2 billion (USD)   
2     3.8   1981.0   1001 to 5000 employees  $100 to $500 million (USD)   
3     3.5   2000.0    501 to 1000 employees  $100 to $500 million (USD)   
4     2.9   1998.0      51 to 200 employees                     Unknown   
5     4.2   2010.0      51 to 200 employees                     Unknown   
6     3.9   1996.0         10000+ employees          $10+ billion (USD)   
7     3.5   1990.0   1001 to 5000 employees      $1 to $2 billion (USD)   
8     4.4   1983.0  5001 to 10000 employees      $2 to $5 b

In [13]:
df_clean.head()

,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,has_competitors
0,0,Sr Data Scientist,$137K-$171K (Glassdoor est.),Description\r\n\r\nThe Senior Data Scientist i...,3.1,Healthfirst\r\n3.1,"New York, NY","New York, NY",1001 to 5000 employees,1993.0,Nonprofit Organization,Insurance Carriers,Insurance,Unknown,1
1,1,Data Scientist,$137K-$171K (Glassdoor est.),"Secure our Nation, Ignite your Future\r\n\r\nJ...",4.2,ManTech\r\n4.2,"Chantilly, VA","Herndon, VA",5001 to 10000 employees,1968.0,Company - Public,Research & Development,Business Services,$1 to $2 billion (USD),0
2,2,Data Scientist,$137K-$171K (Glassdoor est.),Overview\r\n\r\n\r\nAnalysis Group is one of t...,3.8,Analysis Group\r\n3.8,"Boston, MA","Boston, MA",1001 to 5000 employees,1981.0,Private Practice / Firm,Consulting,Business Services,$100 to $500 million (USD),0
3,3,Data Scientist,$137K-$171K (Glassdoor est.),JOB DESCRIPTION:\r\n\r\nDo you have a passion ...,3.5,INFICON\r\n3.5,"Newton, MA","Bad Ragaz, Switzerland",501 to 1000 employees,2000.0,Company - Public,Electrical & Electronic Manufacturing,Manufacturing,$100 to $500 million (USD),1
4,4,Data Scientist,$137K-$171K (Glassdoor est.),Data Scientist\r\nAffinity Solutions / Marketi...,2.9,Affinity Solutions\r\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998.0,Company - Private,Advertising & Marketing,Business Services,Unknown,1


In [14]:
print(f"unique values of size :{df_clean['Size'].unique()}")
print(f"unique values of revenue :{df_clean['Revenue'].unique()}")


unique values of size :<StringArray>
[ '1001 to 5000 employees', '5001 to 10000 employees',
   '501 to 1000 employees',     '51 to 200 employees',
        '10000+ employees',    '201 to 500 employees',
       '1 to 50 employees',                 'Unknown']
Length: 8, dtype: str
unique values of revenue :<StringArray>
[                         'Unknown',           '$1 to $2 billion (USD)',
       '$100 to $500 million (USD)',               '$10+ billion (USD)',
           '$2 to $5 billion (USD)', '$500 million to $1 billion (USD)',
          '$5 to $10 billion (USD)',         '$10 to $25 million (USD)',
         '$25 to $50 million (USD)',        '$50 to $100 million (USD)',
           '$1 to $5 million (USD)',          '$5 to $10 million (USD)',
       'Less than $1 million (USD)']
Length: 13, dtype: str


## NLP Cleaning 

### Job description cleaning 

In [ ]:
import re

EEO_CUTOFF_PATTERNS = [
    # EEO / legal
    r"equal opportunity\/?affirmative action employer",
    r"we are an equal opportunity employer",
    r"equal opportunity employer",
    r"eeo law poster",
    r"we do not discriminate",
    r"if you have a disability under the americans with disability",
    r"applicants and employees are considered for positions",
    r"all qualified applicants will receive consideration for employment",

    # Company branding / culture blocks
    r"our culture is shaped by",
    r"our core values",
    r"earn a referral bonus",
    r"from complexity to clarity",          
]

def clean_for_embedding(text: str) -> str:
    if pd.isna(text):
        return ""

    # 1. Remove XML/HTML artifacts
    text = re.sub(r"<!\[CDATA\[.*?\]\]>", " ", text, flags=re.DOTALL)
    text = re.sub(r"\]\]>", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)

    # 2. Remove URLs and emails
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    # 3. Remove non-ASCII
    text = text.encode("ascii", errors="ignore").decode()

    # 4. Cut at the EARLIEST boilerplate trigger (EEO or company branding)
    earliest_cut = len(text)
    for pattern in EEO_CUTOFF_PATTERNS:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            earliest_cut = min(earliest_cut, match.start())
    text = text[:earliest_cut].strip()

    # 5. Remove duplicate paragraphs
    paragraphs = [p.strip() for p in re.split(r'\n{2,}', text) if p.strip()]
    seen = []
    for p in paragraphs:
        if p not in seen:
            seen.append(p)
    text = " ".join(seen)

    # 6. Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text
df_clean['Job Description'] = df_clean['Job Description'].apply(clean_for_embedding)

### clean company name : 

In [16]:
def clean_company_name(name: str) -> str:
    """
    Removes the embedded rating appended by Glassdoor.
    e.g. 'Healthfirst\\n3.1' → 'Healthfirst'
         'Intuit - Data\\n4.4' → 'Intuit - Data'
    """
    if pd.isna(name):
        return name
    # Split on newline and take only the company name part
    cleaned = name.split("\n")[0].strip()
    return cleaned
df_clean['Company Name'] = df_clean['Company Name'].apply(clean_company_name)

### clean Job title : 

In [17]:
def clean_job_title(title: str) -> str:
    """
    Lightly normalizes job titles:
      - Lowercases
      - Removes special characters (keeps hyphens for terms like 'full-stack')
      - Strips extra whitespace
      - Removes location hints that appear in some titles (e.g. '- Bay Area, CA')
    """
    if pd.isna(title):
        return title

    # Remove location suffixes like "- Bay Area, CA" or "- San Antonio OR"
    title = re.sub(r"\s*[-–]\s*[A-Z][a-zA-Z\s,]+(?:[A-Z]{2})\s*$", "", title)

    # Lowercase
    title = title.lower().strip()

    # Remove special chars except hyphens and slashes (meaningful in titles)
    title = re.sub(r"[^a-z0-9\s\-/]", " ", title)

    # Collapse whitespace
    title = re.sub(r"\s+", " ", title).strip()

    return title

SENIORITY_MAP = {
    r"\bsr\.?\b|\bsenior\b":         "Senior",
    r"\bjr\.?\b|\bjunior\b":         "Junior",
    r"\blead\b|\bstaff\b":           "Lead",
    r"\bprincipal\b":                "Principal",
    r"\bhead\b|\bdirector\b":        "Director",
    r"\bvp\b|\bvice president\b":    "VP",
    r"\bmanager\b|\bmgr\.?\b":       "Manager",
    r"\bentry[\s-]level\b|\bjr\b":   "Junior",
}

def extract_seniority(title: str) -> str:
    """
    Extracts a seniority label from the raw job title.
    Returns one of: Senior, Junior, Lead, Principal, Director, VP, Manager, Mid-level
    """
    if pd.isna(title):
        return "Mid-level"
    title_lower = title.lower()
    for pattern, label in SENIORITY_MAP.items():
        if re.search(pattern, title_lower):
            return label
    return "Mid-level"

df_clean['Job Title'] = df_clean['Job Title'].apply(clean_job_title)
df_clean['Seniority'] = df_clean['Job Title'].apply(extract_seniority)


### Handeling Revenue and Size columns 

In [18]:
print(f"unique values of size :{df_clean['Size'].unique()}")
print(f"unique values of revenue :{df_clean['Revenue'].unique()}")


unique values of size :<StringArray>
[ '1001 to 5000 employees', '5001 to 10000 employees',
   '501 to 1000 employees',     '51 to 200 employees',
        '10000+ employees',    '201 to 500 employees',
       '1 to 50 employees',                 'Unknown']
Length: 8, dtype: str
unique values of revenue :<StringArray>
[                         'Unknown',           '$1 to $2 billion (USD)',
       '$100 to $500 million (USD)',               '$10+ billion (USD)',
           '$2 to $5 billion (USD)', '$500 million to $1 billion (USD)',
          '$5 to $10 billion (USD)',         '$10 to $25 million (USD)',
         '$25 to $50 million (USD)',        '$50 to $100 million (USD)',
           '$1 to $5 million (USD)',          '$5 to $10 million (USD)',
       'Less than $1 million (USD)']
Length: 13, dtype: str


### categorize size 

In [19]:
SIZE_ORDINAL = {
    "1 to 50 employees":       1,
    "51 to 200 employees":     2,
    "201 to 500 employees":    3,
    "501 to 1000 employees":   4,
    "1001 to 5000 employees":  5,
    "5001 to 10000 employees": 6,
    "10000+ employees":        7,   # open-ended upper bound flagged separately
    "Unknown":                 np.nan,
}

SIZE_MIDPOINT = {
    "1 to 50 employees":       25,     # midpoint of [1, 50]
    "51 to 200 employees":     125,    # midpoint of [51, 200]
    "201 to 500 employees":    350,    # midpoint of [201, 500]
    "501 to 1000 employees":   750,    # midpoint of [501, 1000]
    "1001 to 5000 employees":  3000,   # midpoint of [1001, 5000]
    "5001 to 10000 employees": 7500,   # midpoint of [5001, 10000]
    "10000+ employees":        np.nan, # no upper bound → don't fabricate a midpoint
    "Unknown":                 np.nan,
}


def encode_size(size: str) -> tuple:
    """
    Returns (ordinal_rank, midpoint_employees, is_top_size) for a Size string.

    ordinal_rank  → 1–7, useful for tree-based / ordinal models
    midpoint      → estimated employee count; NaN for '10000+' and 'Unknown'
    is_top_size   → binary flag for '10000+ employees' (open-ended bucket)
    """
    size = str(size).strip()
    ordinal   = SIZE_ORDINAL.get(size, np.nan)
    midpoint  = SIZE_MIDPOINT.get(size, np.nan)
    is_top    = 1 if size == "10000+ employees" else 0
    return ordinal, midpoint, is_top
df_clean['Size_Ordinal'], df_clean['Size_Midpoint'], df_clean['Size_is_top'] = zip(*df_clean['Size'].apply(encode_size))

### Categorise Revenue 

In [20]:
REVENUE_ORDINAL = {
    "Less than $1 million (USD)":        1,
    "$1 to $5 million (USD)":            2,
    "$5 to $10 million (USD)":           3,
    "$10 to $25 million (USD)":          4,
    "$25 to $50 million (USD)":          5,
    "$50 to $100 million (USD)":         6,
    "$100 to $500 million (USD)":        7,
    "$500 million to $1 billion (USD)":  8,
    "$1 to $2 billion (USD)":            9,
    "$2 to $5 billion (USD)":            10,
    "$5 to $10 billion (USD)":           11,
    "$10+ billion (USD)":                12,  # open-ended upper bound flagged separately
    "Unknown":                           np.nan,
}

REVENUE_MIDPOINT_MILLIONS = {
    "Less than $1 million (USD)":        0.5,   # midpoint of [0, 1]
    "$1 to $5 million (USD)":            3.0,   # midpoint of [1, 5]
    "$5 to $10 million (USD)":           7.5,   # midpoint of [5, 10]
    "$10 to $25 million (USD)":          17.5,  # midpoint of [10, 25]
    "$25 to $50 million (USD)":          37.5,  # midpoint of [25, 50]
    "$50 to $100 million (USD)":         75.0,  # midpoint of [50, 100]
    "$100 to $500 million (USD)":        300.0, # midpoint of [100, 500]
    "$500 million to $1 billion (USD)":  750.0, # midpoint of [500, 1000]
    "$1 to $2 billion (USD)":            1500.0,# midpoint of [1000, 2000]
    "$2 to $5 billion (USD)":            3500.0,# midpoint of [2000, 5000]
    "$5 to $10 billion (USD)":           7500.0,# midpoint of [5000, 10000]
    "$10+ billion (USD)":                np.nan,# no upper bound → don't fabricate a midpoint
    "Unknown":                           np.nan,
}


def encode_revenue(revenue: str) -> tuple:
    """
    Returns (ordinal_rank, midpoint_usd_millions, is_top_revenue) for a Revenue string.

    ordinal_rank     → 1–12, useful for tree-based / ordinal models
    midpoint_M       → estimated revenue in USD millions; NaN for '$10+ billion' and 'Unknown'
    is_top_revenue   → binary flag for '$10+ billion (USD)' (open-ended bucket)
    """
    revenue = str(revenue).strip()
    ordinal    = REVENUE_ORDINAL.get(revenue, np.nan)
    midpoint   = REVENUE_MIDPOINT_MILLIONS.get(revenue, np.nan)
    is_top     = 1 if revenue == "$10+ billion (USD)" else 0
    return ordinal, midpoint, is_top
df_clean['Revenue_Ordinal'], df_clean['Revenue_Midpoint_M'], df_clean['Revenue_is_top'] = zip(*df_clean['Revenue'].apply(encode_revenue))

In [21]:
df_clean.head()

,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,...,Sector,Revenue,has_competitors,Seniority,Size_Ordinal,Size_Midpoint,Size_is_top,Revenue_Ordinal,Revenue_Midpoint_M,Revenue_is_top
0,0,sr data scientist,$137K-$171K (Glassdoor est.),Description The Senior Data Scientist is respo...,3.1,Healthfirst,"New York, NY","New York, NY",1001 to 5000 employees,1993.0,...,Insurance,Unknown,1,Senior,5.0,3000.0,0,NaN,NaN,0
1,1,data scientist,$137K-$171K (Glassdoor est.),"Secure our Nation, Ignite your Future Join the...",4.2,ManTech,"Chantilly, VA","Herndon, VA",5001 to 10000 employees,1968.0,...,Business Services,$1 to $2 billion (USD),0,Mid-level,6.0,7500.0,0,9.0,1500.0,0
2,2,data scientist,$137K-$171K (Glassdoor est.),Overview Analysis Group is one of the largest ...,3.8,Analysis Group,"Boston, MA","Boston, MA",1001 to 5000 employees,1981.0,...,Business Services,$100 to $500 million (USD),0,Mid-level,5.0,3000.0,0,7.0,300.0,0
3,3,data scientist,$137K-$171K (Glassdoor est.),JOB DESCRIPTION: Do you have a passion for Dat...,3.5,INFICON,"Newton, MA","Bad Ragaz, Switzerland",501 to 1000 employees,2000.0,...,Manufacturing,$100 to $500 million (USD),1,Mid-level,4.0,750.0,0,7.0,300.0,0
4,4,data scientist,$137K-$171K (Glassdoor est.),Data Scientist Affinity Solutions / Marketing ...,2.9,Affinity Solutions,"New York, NY","New York, NY",51 to 200 employees,1998.0,...,Business Services,Unknown,1,Mid-level,2.0,125.0,0,NaN,NaN,0


### Process Salary :

In [22]:
def process_salary(df: pd.DataFrame, column_name: str, audit: bool = True) -> pd.Series:
    """
    Audits and parses a salary column in one pass.

    Audit (printed when audit=True):
      - Detects rows with M (millions), B (billions), unexpected symbols
      - Prints a summary report with examples

    Parsing:
      - Handles K (thousands) and M (millions) multipliers
      - Extracts midpoint from ranges like '$137K-$171K (Glassdoor est.)' → 154000.0
      - Returns a pd.Series of floats (None for unparseable rows)

    Parameters
    ----------
    df          : input DataFrame
    column_name : name of the salary column
    audit       : whether to print the audit report (default True)

    Returns
    -------
    pd.Series of parsed salary floats, aligned to df index
    """

    unexpected_pattern = r"[^0-9\$KkMmBb\-\s\.\(\)a-zA-Z]"

    # ---------- AUDIT ----------
    if audit:
        million_rows  = df[df[column_name].str.contains(r'M', na=False, case=False)]
        billion_rows  = df[df[column_name].str.contains(r'B', na=False, case=False)]
        strange_rows  = df[df[column_name].str.contains(unexpected_pattern, na=False)]

        print(f"--- Salary Audit Report for '{column_name}' ---")
        print(f"Total rows           : {len(df)}")
        print(f"Rows with M (millions): {len(million_rows)}")
        print(f"Rows with B (billions): {len(billion_rows)}")
        print(f"Rows with unexpected symbols: {len(strange_rows)}")

        if not strange_rows.empty:
            print("\nExamples of strange entries:")
            print(strange_rows[column_name].head(5).values)
        print()

    # ---------- PARSE ----------
    def _parse_single(salary_str: str) -> float | None:
        if not isinstance(salary_str, str) or salary_str.lower() == "nan":
            return None
        try:
            upper = salary_str.upper()
            if "M" in upper:
                multiplier = 1_000_000
            else:
                multiplier = 1_000  # default: K

            numbers = re.findall(r"\d+\.?\d*", salary_str)

            if len(numbers) >= 2:
                return (float(numbers[0]) + float(numbers[1])) * multiplier / 2
            elif len(numbers) == 1:
                return float(numbers[0]) * multiplier
            return None
        except Exception:
            return None

    return df[column_name].apply(_parse_single)

df_clean['Salary_cleaned']=process_salary(df_clean, "Salary Estimate", audit=True)


--- Salary Audit Report for 'Salary Estimate' ---
Total rows           : 672
Rows with M (millions): 20
Rows with B (billions): 0
Rows with unexpected symbols: 0



In [23]:
df_clean.head()

,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,...,Revenue,has_competitors,Seniority,Size_Ordinal,Size_Midpoint,Size_is_top,Revenue_Ordinal,Revenue_Midpoint_M,Revenue_is_top,Salary_cleaned
0,0,sr data scientist,$137K-$171K (Glassdoor est.),Description The Senior Data Scientist is respo...,3.1,Healthfirst,"New York, NY","New York, NY",1001 to 5000 employees,1993.0,...,Unknown,1,Senior,5.0,3000.0,0,NaN,NaN,0,154000.0
1,1,data scientist,$137K-$171K (Glassdoor est.),"Secure our Nation, Ignite your Future Join the...",4.2,ManTech,"Chantilly, VA","Herndon, VA",5001 to 10000 employees,1968.0,...,$1 to $2 billion (USD),0,Mid-level,6.0,7500.0,0,9.0,1500.0,0,154000.0
2,2,data scientist,$137K-$171K (Glassdoor est.),Overview Analysis Group is one of the largest ...,3.8,Analysis Group,"Boston, MA","Boston, MA",1001 to 5000 employees,1981.0,...,$100 to $500 million (USD),0,Mid-level,5.0,3000.0,0,7.0,300.0,0,154000.0
3,3,data scientist,$137K-$171K (Glassdoor est.),JOB DESCRIPTION: Do you have a passion for Dat...,3.5,INFICON,"Newton, MA","Bad Ragaz, Switzerland",501 to 1000 employees,2000.0,...,$100 to $500 million (USD),1,Mid-level,4.0,750.0,0,7.0,300.0,0,154000.0
4,4,data scientist,$137K-$171K (Glassdoor est.),Data Scientist Affinity Solutions / Marketing ...,2.9,Affinity Solutions,"New York, NY","New York, NY",51 to 200 employees,1998.0,...,Unknown,1,Mid-level,2.0,125.0,0,NaN,NaN,0,154000.0


### job title processing 

In [24]:
df_clean['Job Title'].unique()

<StringArray>
[                                               'sr data scientist',
                                                   'data scientist',
                         'data scientist / machine learning expert',
                                 'staff data scientist - analytics',
                         'data scientist - statistics early career',
                                                     'data modeler',
                                       'experienced data scientist',
                                        'data scientist - contract',
                                                  'data analyst ii',
                                            'medical lab scientist',
 ...
 'information systems engineering specialist engineering scientist',
               'scientist/research associate-metabolic engineering',
           'vice president biometrics and clinical data management',
   'enterprise data analyst enterprise portfolio management office',
               

In [25]:
# 1. Extract seniority BEFORE cleaning (already done: seniority_level column)
# 2. Extract the core role
ROLE_MAP = {
    r"data scien":        "Data Scientist",
    r"data engineer":     "Data Engineer",
    r"data analyst":      "Data Analyst",
    r"machine learning|ml engineer": "ML Engineer",
    r"data modeler":      "Data Modeler",
    r"research scien":    "Research Scientist",
    r"business intel":    "BI Analyst",
    r"statistician":      "Statistician",
}

def extract_core_role(title: str) -> str:
    if pd.isna(title):
        return "Other"
    title_lower = title.lower()
    for pattern, role in ROLE_MAP.items():
        if re.search(pattern, title_lower):
            return role
    return "Other"

df_clean["job_title"] = df_clean["Job Title"].apply(extract_core_role)

In [26]:
df_clean['job_title'].unique()

<StringArray>
[    'Data Scientist',       'Data Modeler',       'Data Analyst',
              'Other',         'BI Analyst',      'Data Engineer',
        'ML Engineer', 'Research Scientist']
Length: 8, dtype: str

In [27]:
df_clean.head()

,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,...,has_competitors,Seniority,Size_Ordinal,Size_Midpoint,Size_is_top,Revenue_Ordinal,Revenue_Midpoint_M,Revenue_is_top,Salary_cleaned,job_title
0,0,sr data scientist,$137K-$171K (Glassdoor est.),Description The Senior Data Scientist is respo...,3.1,Healthfirst,"New York, NY","New York, NY",1001 to 5000 employees,1993.0,...,1,Senior,5.0,3000.0,0,NaN,NaN,0,154000.0,Data Scientist
1,1,data scientist,$137K-$171K (Glassdoor est.),"Secure our Nation, Ignite your Future Join the...",4.2,ManTech,"Chantilly, VA","Herndon, VA",5001 to 10000 employees,1968.0,...,0,Mid-level,6.0,7500.0,0,9.0,1500.0,0,154000.0,Data Scientist
2,2,data scientist,$137K-$171K (Glassdoor est.),Overview Analysis Group is one of the largest ...,3.8,Analysis Group,"Boston, MA","Boston, MA",1001 to 5000 employees,1981.0,...,0,Mid-level,5.0,3000.0,0,7.0,300.0,0,154000.0,Data Scientist
3,3,data scientist,$137K-$171K (Glassdoor est.),JOB DESCRIPTION: Do you have a passion for Dat...,3.5,INFICON,"Newton, MA","Bad Ragaz, Switzerland",501 to 1000 employees,2000.0,...,1,Mid-level,4.0,750.0,0,7.0,300.0,0,154000.0,Data Scientist
4,4,data scientist,$137K-$171K (Glassdoor est.),Data Scientist Affinity Solutions / Marketing ...,2.9,Affinity Solutions,"New York, NY","New York, NY",51 to 200 employees,1998.0,...,1,Mid-level,2.0,125.0,0,NaN,NaN,0,154000.0,Data Scientist


In [28]:
df_clean.columns

Index(['index', 'Job Title', 'Salary Estimate', 'Job Description', 'Rating',
       'Company Name', 'Location', 'Headquarters', 'Size', 'Founded',
       'Type of ownership', 'Industry', 'Sector', 'Revenue', 'has_competitors',
       'Seniority', 'Size_Ordinal', 'Size_Midpoint', 'Size_is_top',
       'Revenue_Ordinal', 'Revenue_Midpoint_M', 'Revenue_is_top',
       'Salary_cleaned', 'job_title'],
      dtype='str')

In [29]:
df_clean['Company Name'].unique()

<StringArray>
[              'Healthfirst',                   'ManTech',
            'Analysis Group',                   'INFICON',
        'Affinity Solutions',               'HG Insights',
                  'Novartis',                    'iRobot',
             'Intuit - Data',        'XSELL Technologies',
 ...
              'Pactera Edge',       'Qurate Retail Group',
 'A-Line Staffing Solutions',       'Clear Ridge Defense',
   'Criterion Systems, Inc.',       'Foundation Medicine',
                  'TRANZACT',                      'JKGT',
                'AccessHope',      'ChaTeck Incorporated']
Length: 432, dtype: str

### treatement of location and headquarters

In [30]:
locations=df_clean['Location'].unique()
print(locations)

<StringArray>
[     'New York, NY',     'Chantilly, VA',        'Boston, MA',
        'Newton, MA', 'Santa Barbara, CA',     'Cambridge, MA',
       'Bedford, MA',     'San Diego, CA',       'Chicago, IL',
       'Herndon, VA',
 ...
        'Orange, CA',    'Bridgeport, WV',      'Oakville, CA',
    'Naperville, IL',       'Houston, TX',       'Redmond, WA',
  'West Chester, PA',      'Quantico, VA',      'Fort Lee, NJ',
     'Irwindale, CA']
Length: 207, dtype: str


In [31]:
df_clean['Headquarters'].unique()

<StringArray>
[          'New York, NY',            'Herndon, VA',             'Boston, MA',
 'Bad Ragaz, Switzerland',      'Santa Barbara, CA',     'Basel, Switzerland',
            'Bedford, MA',      'Mountain View, CA',            'Chicago, IL',
            'Mc Lean, VA',
 ...
            'Langley, VA',              'Plano, TX',        'Albertville, AL',
             'Orange, CA',          'Littleton, CO',       'Oakville, Canada',
          'San Bruno, CA',       'West Chester, PA',              'Utica, MI',
           'Fort Lee, NJ']
Length: 229, dtype: str

In [32]:
def parse_location(loc: str) -> tuple:
    """Returns (city, state) from 'City, ST' format."""
    if pd.isna(loc) or loc in ("Unknown", "-1"):
        return np.nan, np.nan
    parts = [p.strip() for p in loc.split(",")]
    if len(parts) >= 2:
        return parts[0], parts[1]
    return parts[0], np.nan   # state-only or single token like "Texas"

# Parse both columns
df[["job_city", "job_state"]]  = df["Location"].apply(lambda x: pd.Series(parse_location(x)))
df[["hq_city",  "hq_state"]]   = df["Headquarters"].apply(lambda x: pd.Series(parse_location(x)))

# Binary: is the job at HQ or remote?
df["is_remote"]    = df["Location"].str.lower().eq("remote").astype(int)
df["is_at_hq"]     = (df["Location"] == df["Headquarters"]).astype(int)

# Binary: is HQ domestic or international?
US_STATES = { "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL",
              "IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT",
              "NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI",
              "SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC" }
df["hq_is_international"] = df["hq_state"].apply(
    lambda s: 0 if pd.isna(s) or s in US_STATES else 1
)

In [33]:
df_clean[["job_city", "job_state"]] = df_clean["Location"].apply(lambda x: pd.Series(parse_location(x)))
df_clean[["hq_city", "hq_state"]]   = df_clean["Headquarters"].apply(lambda x: pd.Series(parse_location(x)))

df_clean["is_remote"]           = df_clean["Location"].str.lower().eq("remote").astype(int)
df_clean["is_at_hq"]            = (df_clean["Location"] == df_clean["Headquarters"]).astype(int)
df_clean["hq_is_international"] = df_clean["hq_state"].apply(
    lambda s: 0 if pd.isna(s) or s in US_STATES else 1
)

In [36]:
df_clean.columns

Index(['index', 'Job Title', 'Salary Estimate', 'Job Description', 'Rating',
       'Company Name', 'Location', 'Headquarters', 'Size', 'Founded',
       'Type of ownership', 'Industry', 'Sector', 'Revenue', 'has_competitors',
       'Seniority', 'Size_Ordinal', 'Size_Midpoint', 'Size_is_top',
       'Revenue_Ordinal', 'Revenue_Midpoint_M', 'Revenue_is_top',
       'Salary_cleaned', 'job_title', 'job_city', 'job_state', 'hq_city',
       'hq_state', 'is_remote', 'is_at_hq', 'hq_is_international'],
      dtype='str')

### type of ownership treatement : 

In [37]:
df_clean['Type of ownership'].unique()

<StringArray>
[        'Nonprofit Organization',               'Company - Public',
        'Private Practice / Firm',              'Company - Private',
                     'Government', 'Subsidiary or Business Segment',
             'Other Organization',                        'Unknown',
                       'Hospital',                  'Self-employed',
           'College / University',                       'Contract']
Length: 12, dtype: str

In [ ]:
OWNERSHIP_MAP = {
    "Company - Private":           "Private",
    "Company - Public":            "Public",
    "Subsidiary or Business Segment": "Private",   # still privately structured
    "Private Practice / Firm":     "Private",
    "Nonprofit Organization":      "Nonprofit",
    "Government":                  "Government",
    "College / University":        "Government",   # publicly funded institution
    "Hospital":                    "Nonprofit",    # most US hospitals are nonprofit
    "Self-employed":               "Other",
    "Contract":                    "Other",
    "Other Organization":          "Other",
    "Unknown":                     "Unknown",
}

df_clean["type_of_ownership"] = df_clean["Type of ownership"].map(OWNERSHIP_MAP)

In [ ]:
df_clean['type_of_ownership'].unique()

<StringArray>
['Nonprofit', 'Public', 'Private', 'Government', 'Other', 'Unknown']
Length: 6, dtype: str

### Handeling sector and industry : 

In [42]:
df_clean['Sector'].unique()

<StringArray>
[                         'Insurance',                  'Business Services',
                      'Manufacturing',             'Information Technology',
          'Biotech & Pharmaceuticals',                             'Retail',
       'Oil, Gas, Energy & Utilities',                         'Government',
                        'Health Care',                            'Finance',
                'Aerospace & Defense',                            'Unknown',
         'Transportation & Logistics',                              'Media',
                 'Telecommunications',                        'Real Estate',
                   'Travel & Tourism',             'Agriculture & Forestry',
                          'Education',                 'Accounting & Legal',
                         'Non-Profit', 'Construction, Repair & Maintenance',
                  'Consumer Services']
Length: 23, dtype: str

In [43]:
df_clean['Industry'].unique()

<StringArray>
[                      'Insurance Carriers',
                   'Research & Development',
                               'Consulting',
    'Electrical & Electronic Manufacturing',
                  'Advertising & Marketing',
             'Computer Hardware & Software',
                'Biotech & Pharmaceuticals',
 'Consumer Electronics & Appliances Stores',
  'Enterprise Software & Network Solutions',
                              'IT Services',
                                   'Energy',
                   'Chemical Manufacturing',
                         'Federal Agencies',
                                 'Internet',
         'Health Care Services & Hospitals',
    'Investment Banking & Asset Management',
                      'Aerospace & Defense',
                                'Utilities',
                                  'Unknown',
                'Express Delivery Services',
                   'Staffing & Outsourcing',
          'Insurance Agencies & Brokerage

In [ ]:
df_clean.columns
df_clean.drop(columns=["Job Title","","Size", "Revenue"], inplace=True)

Index(['index', 'Job Title', 'Salary Estimate', 'Job Description', 'Rating',
       'Company Name', 'Location', 'Headquarters', 'Size', 'Founded',
       'Type of ownership', 'Industry', 'Sector', 'Revenue', 'has_competitors',
       'Seniority', 'Size_Ordinal', 'Size_Midpoint', 'Size_is_top',
       'Revenue_Ordinal', 'Revenue_Midpoint_M', 'Revenue_is_top',
       'Salary_cleaned', 'job_title', 'job_city', 'job_state', 'hq_city',
       'hq_state', 'is_remote', 'is_at_hq', 'hq_is_international',
       'ownership_grouped'],
      dtype='str')